# Day 23: Lab - Data Wrangling Mini-Project

## Project Goal
This lab combines several tools from this week — string cleaning, datetime conversion, MultiIndex grouping, and rolling statistics — into one realistic mini-workflow: taking a messy raw sales log and turning it into something analyzable. Real logged data is rarely clean on arrival (inconsistent casing, text dates, duplicate-looking entries that are actually the same customer) and wrangling it into shape is a huge and unavoidable part of actual data science work, often taking more time than the modeling itself. The steps below mirror a typical wrangling sequence: inspect the raw data, clean it, aggregate it meaningfully, then look for trends. Pay attention to how each step builds on the previous one — cleaning has to happen before aggregating, and aggregating before computing a rolling trend.</cell id="cell-1">


In [ ]:
import pandas as pd
import numpy as np

# intentionally messy: inconsistent email casing, order_date is still text, not datetime
raw = pd.DataFrame({
    'order_date': ['2024-01-01', '2024-01-02', '2024-01-02', '2024-01-03', '2024-01-04'],
    'customer_email': ['Amy@Gmail.com', 'ben@YAHOO.com', 'amy@gmail.com', 'cara@outlook.com', 'ben@yahoo.com'],
    'region': ['East', 'East', 'East', 'West', 'East'],
    'amount': [250, 180, 300, 420, 150]
})
print(raw)

## Step 1: Clean the Data
Before any meaningful aggregation, the raw inconsistencies need fixing. Notice that "Amy@Gmail.com" and "amy@gmail.com" in the raw data are actually the same customer, just typed with different casing — if left as-is, they'd be treated as two different people in any grouping or count, silently corrupting the analysis. Lowercasing the email column (as covered on Day 20) fixes that, and converting `order_date` to a real datetime type (as covered on Day 21) is required before we can group by date or compute date-based rolling stats later. This is a textbook example of why cleaning always comes before analysis — skipping it here would make Steps 2 and 3 both technically run but produce wrong answers.</cell id="cell-3">


In [ ]:
# lowercase emails so "Amy@Gmail.com" and "amy@gmail.com" are recognized as the same customer
raw['customer_email'] = raw['customer_email'].str.lower()
raw['order_date'] = pd.to_datetime(raw['order_date'])  # convert text dates to real datetime values
print(raw)

## Step 2: Aggregate by Region and Date
With clean data in hand, we can summarize it meaningfully. Grouping by two columns at once (`region` and `order_date`) and summing the `amount` produces a MultiIndex Series (as covered on Day 19), where each value is the total sales for one region on one specific day. This kind of two-level aggregation is exactly what you'd build for a sales report broken down by both location and time — it answers "how much did each region sell, and on which days?" in one structure, rather than needing separate tables per region. In the output, look for how rows sharing the same region but different dates appear consecutively under that region's label.</cell id="cell-5">


In [ ]:
# group by region AND date, then sum amounts within each group -> MultiIndex Series (region, order_date)
summary = raw.groupby(['region', 'order_date'])['amount'].sum()
print(summary)

## Step 3: Rolling Trend
Finally, we look at the overall trend over time, independent of region. First we collapse the data down to one total per day (summing across all regions), producing a simple time-indexed Series — the kind we worked with on Day 15 and Day 22. Then a 2-day rolling average smooths that daily total, making it easier to spot whether sales are generally trending up or down rather than jumping around from a single unusually high or low day. In the output, expect the first day to show `NaN` (no prior day to average with), and each following value to be the average of that day's total and the day before it.</cell id="cell-7">


In [ ]:
# collapse to one total-sales value per day (summed across regions)
daily = raw.groupby('order_date')['amount'].sum()
print(daily.rolling(2).mean())  # 2-day rolling average of daily totals, to smooth the trend